# Section 5.4 / 5.5 — 從既有 pkl 產出對照表

- **5.4 Representation / DR Comparison**：Mapper(TDA) vs one-hot vs PCA vs UMAP vs MCA
- **5.5 Ablation Study**：逐一 toggle 元件，看貢獻

資料來源：
- TDA 子群 / full_data / dummy / onehot → `ModelPerformanceSeed/{encoding}/{algo}/{group}.pkl`（新版：時間切分＋無洩漏）
- PCA / UMAP / MCA → `../CompareOther/{algo}/{pca|umap|mca_only}_{algo}.pkl`（現成）
- 舊隨機切分 → `ModelPerformanceSeed/{seed}/{algo}/{group}.pkl`（用於 5.5 的 time vs random 那列）

> **門檻**：固定機率 0.5 會讓 recall 很低（模型有加權、測試集又平衡）。設 `THRESHOLD='youden'` 用 ROC 上最大化 TPR−FPR 的門檻，recall/f1 會合理很多。注意 Youden 是在測試集上找門檻，略樂觀；嚴格做法是用訓練/驗證分數選門檻再套到測試集。roc_auc / pr_auc 與門檻無關，是最公平的比較欄。

> **測試集分布**：重跑後 pkl 的測試集已改為**保留真實不平衡分布**（不再砍成 50/50）。因此欄位順序以 **PR-AUC / ROC-AUC / balanced_acc / recall** 為主；**precision / F1 / accuracy 是真實分布下的值（會比舊的平衡版低，但更誠實）**，放在後面。PR-AUC 是極度不平衡下最該當主指標的欄。

In [5]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from utils.evaluate import table_5_4, table_5_5, aggregate_tda, metrics_from_pkl, SUBGROUPS

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

PERF_BASE = "ModelPerformanceSeed"     # 新版結果根目錄（相對 Models/）
COMPARE_BASE = "../CompareOther"       # PCA/UMAP/MCA 現成結果
ENC = "onehot"                          # 主表用的編碼（onehot / dummy）
THRESHOLD = 'youden'   # 'fixed'(機率0.5) 或 'youden'(ROC 最佳門檻，改善 recall/f1)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 5.4 Representation / DR Comparison

固定分類器，比較不同「高維類別資料的表徵/處理方式」。每個分類器各一張表。
Mapper 列是 10 個子群 pooled 後的系統級指標；其餘為 full_data 單一全域模型。

> 注意：`CompareOther` 的 PCA/UMAP/MCA 若是用舊流程（隨機切分/SMOTE）產生，與新版時間切分的 Mapper 不是完全 apples-to-apples；要嚴格比較需用相同流程重跑這三個 baseline。

> **門檻**：固定機率 0.5 會讓 recall 很低（模型有加權、測試集又平衡）。設 `THRESHOLD='youden'` 用 ROC 上最大化 TPR−FPR 的門檻，recall/f1 會合理很多。注意 Youden 是在測試集上找門檻，略樂觀；嚴格做法是用訓練/驗證分數選門檻再套到測試集。roc_auc / pr_auc 與門檻無關，是最公平的比較欄。


In [4]:
for algo in ["xgboost", "logistic", "svc"]:
    print(f"\n===== 5.4  algo = {algo}  (encoding={ENC}) =====")
    tbl = table_5_4(algo, perf_base=PERF_BASE, encoding=ENC, compare_base=COMPARE_BASE, threshold=THRESHOLD)
    display(tbl)



===== 5.4  algo = xgboost  (encoding=dummy) =====


,pr_auc,roc_auc,balanced_acc,recall,precision,f1,accuracy,threshold,n_test
method,,,,,,,,,
"Mapper (TDA, dummy)",0.0274,0.7274,0.6816,0.5350,0.0152,0.0295,0.8267,0.1214,66823
One-hot (full_data),0.0137,0.6485,0.6229,0.4939,0.0097,0.0191,0.7506,0.0864,66819
PCA (full_data),0.0175,0.7216,0.6635,0.6620,0.0089,0.0176,0.6650,0.0134,94180
UMAP (full_data),0.0129,0.6785,0.6351,0.4178,0.0127,0.0246,0.8504,0.2860,94180
MCA (full_data),0.0260,0.7705,0.7096,0.6925,0.0114,0.0224,0.7266,0.0193,94180



===== 5.4  algo = logistic  (encoding=dummy) =====


,pr_auc,roc_auc,balanced_acc,recall,precision,f1,accuracy,threshold,n_test
method,,,,,,,,,
"Mapper (TDA, dummy)",0.0225,0.7088,0.6594,0.4559,0.0162,0.0313,0.8609,0.5991,66823
One-hot (full_data),0.0277,0.7115,0.6674,0.5701,0.0118,0.0231,0.7637,0.7771,66819
PCA (full_data),0.0137,0.7002,0.6555,0.5869,0.0096,0.0188,0.7234,0.5355,94180
UMAP (full_data),0.0102,0.6718,0.6468,0.5047,0.0107,0.0211,0.7877,0.5483,94180
MCA (full_data),0.0172,0.7219,0.6886,0.5469,0.0144,0.0281,0.8290,0.5522,94180



===== 5.4  algo = svc  (encoding=dummy) =====


,pr_auc,roc_auc,balanced_acc,recall,precision,f1,accuracy,threshold,n_test
method,,,,,,,,,
"Mapper (TDA, dummy)",0.0173,0.7252,0.6943,0.6322,0.0127,0.0249,0.7558,-0.1049,66823
One-hot (full_data),0.0333,0.7224,0.6619,0.5274,0.0126,0.0246,0.7950,0.1403,66819
PCA (full_data),0.0137,0.7004,0.6541,0.5962,0.0093,0.0183,0.7114,0.1107,94180
UMAP (full_data),0.0096,0.6619,0.6328,0.6479,0.0076,0.0151,0.6179,-0.2237,94180
MCA (full_data),0.0167,0.7104,0.6804,0.5469,0.0132,0.0257,0.8126,0.0912,94180
